In [1]:
import json
from collections import OrderedDict
import os
import numpy as np
from tqdm import tqdm
import SimpleITK as sitk

# Load preprocessed metadata

In [2]:
root = '/workspace/data/preprocessed'
img_root = os.path.join(root, 'B50')

with open(os.path.join(root, 'distortion_pivot.json'), 'r') as f:
    distortion_pivot = json.load(f)
with open(os.path.join(root, 'data_split.json'), 'r') as f:
    data_split = json.load(f)
with open(os.path.join(root, 'crop_coor.json'), 'r') as f:
    crop_coor = json.load(f)
with open(os.path.join(root, 'distortion_severity_labeling.json'), 'r') as f:
    distortion_label = json.load(f)

In [3]:
# Collect all slice filenames from distortion labels
imgs = []
for pat in distortion_label:
    for s in distortion_label[pat].keys():
        imgs.append(pat + '_' + s + '.nii.gz')
len(imgs)

4580

In [ ]:
# Build per-slice metadata and split into train/val/test sets
trainset = []
valset = []
testset = []

train_pats = []
val_pats = []
test_pats = []

test_distortion_0 = []
test_distortion_1 = []
test_distortion_2 = []
test_distortion_3 = []
test_distortion_4 = []

for img in tqdm(imgs):
    my_data = OrderedDict()
    my_data['image'] = img
    my_data['distortion_pivot'] = distortion_pivot[img.split('.')[0]]
    my_data['distortion_severity'] = int(distortion_label["_".join(img.split('.')[0].split('_')[:-1])][img.split('.')[0].split('_')[-1]])

    # Skip slices with constant or near-constant ADC signal
    img_path = os.path.join(root, "ADC", img)
    img_arr = sitk.GetArrayFromImage(sitk.ReadImage(img_path))

    if (img_arr.min() == img_arr.max()) or (img_arr.std() < 0.1):
        print(f"Skip {img_path}, low variance")
        continue

    pat_id = "_".join(img.split("_")[:2])
    my_data['crop_coor'] = crop_coor[pat_id]

    img_name = "_".join(img.split('.')[0].split('_')[:-1])

    if img_name in data_split['train']:
        trainset.append(my_data)
        train_pats.append(img_name)
    elif img_name in data_split['val']:
        valset.append(my_data)
        val_pats.append(img_name)
    elif img_name in data_split['test']:
        testset.append(my_data)
        test_pats.append(img_name)

        # Group test samples by distortion severity for analysis
        severity = my_data['distortion_severity']
        if severity == 0:
            test_distortion_0.append(my_data)
        elif severity == 1:
            test_distortion_1.append(my_data)
        elif severity == 2:
            test_distortion_2.append(my_data)
        elif severity == 3:
            test_distortion_3.append(my_data)
        elif severity == 4:
            test_distortion_4.append(my_data)
        else:
            print("Invalid distortion severity:", severity)

In [ ]:
len(set(train_pats)), len(set(val_pats)), len(set(test_pats))

In [ ]:
len(test_distortion_0), len(test_distortion_1), len(test_distortion_2), len(test_distortion_3), len(test_distortion_4)

In [14]:
# Export train/val/test metadata as JSONL files
for split_name, dataset in [("train", trainset), ("validation", valset), ("test", testset)]:
    jsonl_path = os.path.join(root, f"PROSTATEx_{split_name}_metadata.jsonl")
    with open(jsonl_path, "w") as f:
        for metadata in dataset:
            json.dump(metadata, f)
            f.write("\n")

In [4]:
plot_target = [
    "ProstateX-0123_01-30-2012_10.nii.gz",
    "ProstateX-0130_02-09-2012_13.nii.gz"
]

In [7]:
plotset = []
for img in plot_target:
    my_data = OrderedDict()
    my_data['image'] = img
    my_data['distortion_pivot'] = distortion_pivot[img.split('.')[0]]
    my_data['distortion_severity'] = int(distortion_label["_".join(img.split('.')[0].split('_')[:-1])][img.split('.')[0].split('_')[-1]])
    pat_id = "_".join(img.split("_")[:2])
    my_data['crop_coor'] = crop_coor[pat_id]
    plotset.append(my_data)

In [8]:
jsonl_file_path = os.path.join(root, "PROSTATEx_plot_metadata.jsonl")
with open(jsonl_file_path, "w") as f:
    for metadata in plotset:
        json.dump(metadata, f)
        f.write("\n")